In [1]:
import os
import json
import math
from PIL import Image
import imagehash
import imageio
from IPython.display import display

## Video Frame Sampling using imagehash

In [3]:
def extract_unique_frames(video_path, output_dir, similarity_threshold=5, interval_seconds=10):
    """
    Args:
        video_path (str): The full path to the input video file.
        output_dir (str): The path to the directory where frames will be saved.
        similarity_threshold (int): The threshold for hash comparison. A lower
                                    value means frames must be more similar to
                                    be considered duplicates. Default is 5.
        interval_seconds (int): The duration of each time segment in seconds.
    """
    # --- 1. Setup and Validation ---
    print(f"Starting frame extraction for '{video_path}'...")

    if not os.path.exists(video_path):
        print(f"Error: Video file not found at '{video_path}'")
        return

    os.makedirs(output_dir, exist_ok=True)
    print(f"Frames will be saved in '{output_dir}'")


    # --- 2. Open Video and Get Metadata ---
    try:
        reader = imageio.get_reader(video_path)
    except Exception as e:
        print(f"Error opening video file with imageio: {e}")
        print("Please ensure FFmpeg is installed and accessible on your system.")
        print("You can often install it by running: pip install imageio[ffmpeg]")
        return
        
    meta_data = reader.get_meta_data()
    fps = meta_data.get('fps', 30)
    
    # Get total video duration directly from metadata
    video_duration = meta_data.get('duration')
    if video_duration is None:
        # Fallback calculation if duration is not in metadata
        video_duration = reader.count_frames() / fps
    print(f"Detected video duration: {video_duration:.2f} seconds.")
    
    # Process one frame per second
    # frame_interval = int(round(fps))
    desired_processing_fps = 5
    frame_interval = int(round(fps / desired_processing_fps))
    if frame_interval < 1: # Ensure we don't divide by zero or go below 1
        frame_interval = 1

    # --- 3. Frame Extraction Loop ---
    saved_frame_count = 0
    last_hash = None
    
    # Use a temporary dict with integer keys for efficiency
    sampled_frame = {}
    
    # Iterate through each frame in the video
    for frame_num, frame in enumerate(reader):
        # --- 4. Process Frame at ~1 FPS Interval ---
        if frame_num % frame_interval == 0:
            pil_img = Image.fromarray(frame)
            current_hash = imagehash.phash(pil_img)

            # --- 5. Check for Uniqueness ---
            if last_hash is None or (current_hash - last_hash) > similarity_threshold:
                filename = f"frame_{saved_frame_count:05d}.png"
                output_path = os.path.join(output_dir, filename)

                imageio.imwrite(output_path, frame)
                
                current_time_sec = frame_num / fps
                print(f"Saved unique frame: {filename} (at video time ~{current_time_sec:.2f}s)")
                sampled_frame[current_time_sec] = filename 
                last_hash = current_hash
                saved_frame_count += 1
    try:
        with open("vid_frames/frame_metadata.json", 'w') as f:
            json.dump(sampled_frame, f, indent=4)
        print(f"\nSuccessfully created metadata ")
    except Exception as e:
        print(f"\nError writing metadata file: {e}")
             
    


video_path = "/home/znyd/hacking/edu-cut/src/pre_processing/downloads/video/vector_3B1B.mp4"
extract_unique_frames(video_path, 'vid_frames', 20)


Starting frame extraction for '/home/znyd/hacking/edu-cut/src/pre_processing/downloads/video/vector_3B1B.mp4'...
Frames will be saved in 'vid_frames'
Detected video duration: 591.16 seconds.
Saved unique frame: frame_00000.png (at video time ~0.00s)
Saved unique frame: frame_00001.png (at video time ~0.20s)
Saved unique frame: frame_00002.png (at video time ~4.20s)
Saved unique frame: frame_00003.png (at video time ~4.60s)
Saved unique frame: frame_00004.png (at video time ~11.20s)
Saved unique frame: frame_00005.png (at video time ~11.80s)
Saved unique frame: frame_00006.png (at video time ~12.20s)
Saved unique frame: frame_00007.png (at video time ~12.40s)
Saved unique frame: frame_00008.png (at video time ~12.60s)
Saved unique frame: frame_00009.png (at video time ~12.80s)
Saved unique frame: frame_00010.png (at video time ~13.00s)
Saved unique frame: frame_00011.png (at video time ~13.40s)
Saved unique frame: frame_00012.png (at video time ~14.20s)
Saved unique frame: frame_00013.p

In [4]:
video_path = "/home/znyd/hacking/edu-cut/src/pre_processing/downloads/video/vector_3B1B.mp4"
try:
    reader = imageio.get_reader(video_path)
except Exception as e:
    print(f"Error opening video file with imageio: {e}")
    print("Please ensure FFmpeg is installed and accessible on your system.")
    print("You can often install it by running: pip install imageio[ffmpeg]")

    
video_duration = reader.get_meta_data()['duration']
video_duration
loop_seg = (int(video_duration) // 10)+1
print(video_duration, loop_seg)

591.16 60


In [5]:
interval_seconds = 10

with open("vid_frames/frame_metadata.json", 'r', encoding='utf-8') as f:
    sampled_frame = json.load(f)

loop_seg = (int(video_duration) // interval_seconds)+1
final_metadata = {}
time_stamps = sorted([float(s_ts) for s_ts in sampled_frame.keys()])

for i in range(loop_seg):
    s_t = i*10
    e_t = min((i+1)*10, int(video_duration))

    time_str = f"{s_t}s-{e_t}s" 
    final_metadata[time_str] = []

    s_t, e_t = float(s_t), float(e_t)

    for ts in time_stamps:
        if ts >= s_t and ts <= e_t:
            ts = str(float(ts)) 
            final_metadata[time_str].append(sampled_frame[ts])

try:
    with open("vid_frames/segment_metadata.json", 'w') as f:
        json.dump(final_metadata, f, indent=4)
    print(f"\nSuccessfully created metadata ")
except Exception as e:
    print(f"\nError writing metadata file: {e}")      


Successfully created metadata 


## Subtitle preprocessing

In [6]:
import pandas as pd

In [7]:
sub_path = "/home/znyd/hacking/edu-cut/src/pre_processing/downloads/transcription_vector_3B1B.csv"
subtitle_df = pd.read_csv(sub_path)
subtitle_df

,Start (s),End (s),Segment
0,10.88,15.28,"The fundamental, root-of-it-all building block..."
1,15.60,19.92,So it's worth making sure that we're all on th...
2,20.24,30.24,"You see, broadly speaking, there are three dis..."
3,30.56,34.48,The physics student perspective is that vector...
4,34.72,43.36,What defines a given vector is its length and ...
...,...,...,...
68,540.16,551.20,"And on the flip side, it gives people like phy..."
69,551.52,562.48,"When I do mathy animations, for example, I sta..."
70,562.80,566.48,And doing that usually relies on a lot of line...
71,567.44,574.80,"So there are your vector basics, and in the ne..."


In [8]:
sub_dict = subtitle_df.to_dict(orient='records')
sub_dict[:5]

[{'Start (s)': 10.88,
  'End (s)': 15.28,
  'Segment': 'The fundamental, root-of-it-all building block for linear algebra is the vector.'},
 {'Start (s)': 15.6,
  'End (s)': 19.92,
  'Segment': "So it's worth making sure that we're all on the same page about what exactly a vector is."},
 {'Start (s)': 20.24,
  'End (s)': 30.24,
  'Segment': "You see, broadly speaking, there are three distinct but related ideas about vectors, which I'll call the physics student perspective, the computer science student perspective, and the mathematician's perspective."},
 {'Start (s)': 30.56,
  'End (s)': 34.48,
  'Segment': 'The physics student perspective is that vectors are arrows pointing in space.'},
 {'Start (s)': 34.72,
  'End (s)': 43.36,
  'Segment': "What defines a given vector is its length and the direction it's pointing, but as long as those two facts are the same, you can move it all around and it's still the same vector."}]

In [9]:
mx = float('-inf') 
for s in sub_dict:
    duration = s['End (s)'] - s["Start (s)"]
    if duration >= mx :
        mx = duration
       
print(mx)

17.039999999999992


In [10]:
sub_metadata = {}
for i in range(loop_seg):
    s_t = i*10
    e_t = min((i+1)*10, int(video_duration))

    time_str = f"{s_t}s-{e_t}s" 
    sub_metadata[time_str] = []

    s_t, e_t = float(s_t), float(e_t)

time_seg = sub_metadata.keys()
for idx, s in enumerate(sub_dict):
    start = math.floor(s['Start (s)'])
    end = math.ceil(s['End (s)'])
    tmp = []
    for t in time_seg:
        st, et = t.split('-')
        st = float(st[:-1])
        et = float(et[:-1])
        if start <= st and st <= end or start <= et and et <= end:
            tmp.append(t)
    for i in tmp:
        sub_metadata[i].append(s['Segment'])
    # print(idx) 
    # print(tmp)
for t in time_seg:
    if len(sub_metadata[t]) == 0:
        sub_metadata[t].append("..Silent/No one is not taking..")
            
# print(sub_metadata)

try:
    with open("segment_subtitle.json", 'w') as f:
        json.dump(sub_metadata, f, indent=4)
    print(f"\nSuccessfully created metadata ")
except Exception as e:
    print(f"\nError writing metadata file: {e}")  


Successfully created metadata 


## Combine Frame and subtitle

In [11]:
def combine_json_data(frames_json_path, subtitles_json_path, output_json_path):
    """
    Combines JSON data from frame extraction and subtitle generation into a
    single structured JSON file.

    Args:
        frames_json_path (str): Path to the JSON file containing frame data.
        subtitles_json_path (str): Path to the JSON file with subtitle data.
        output_json_path (str): Path for the combined output JSON file.
    """
    print("Starting JSON combination process...")

    # --- 1. Read and Load Input JSON Files ---
    try:
        with open(frames_json_path, 'r', encoding='utf-8') as f:
            frames_data = json.load(f)
        print(f"Successfully loaded frames data from '{frames_json_path}'")
    except FileNotFoundError:
        print(f"Error: Frames JSON file not found at '{frames_json_path}'")
        return
    except json.JSONDecodeError:
        print(f"Error: Could not decode JSON from '{frames_json_path}'. Check for formatting errors.")
        return

    try:
        with open(subtitles_json_path, 'r', encoding='utf-8') as f:
            subtitles_data = json.load(f)
        print(f"Successfully loaded subtitles data from '{subtitles_json_path}'")
    except FileNotFoundError:
        print(f"Error: Subtitles JSON file not found at '{subtitles_json_path}'")
        return
    except json.JSONDecodeError:
        print(f"Error: Could not decode JSON from '{subtitles_json_path}'. Check for formatting errors.")
        return

    # --- 2. Combine Data for Each Time Segment ---

    # Get all unique keys (time segments) from both dictionaries
    all_keys = frames_data.keys() 

    # Sort keys chronologically based on the start time (the number before '-')
    # This ensures the final list is in order.
    
    combined_list = []
    for key in all_keys:
        # Create a dictionary for the current segment
        segment_data = {
            "time_stamps": key,
            # Use .get(key, []) to safely get the list, or an empty list if the key doesn't exist
            "frames": frames_data.get(key, []),
            "subtitle": subtitles_data.get(key, [])
        }
        combined_list.append(segment_data)

    # --- 3. Write the Combined List to a New JSON File ---
    try:
        with open(output_json_path, 'w', encoding='utf-8') as f:
            json.dump(combined_list, f, indent=4)
        print(f"\nSuccessfully combined data into '{output_json_path}'")
    except Exception as e:
        print(f"\nAn error occurred while writing the output file: {e}")


if __name__ == "__main__":
    # Define file paths for our dummy data
    frames_file = "/home/znyd/hacking/edu-cut/src/pre_processing/vid_frames/segment_metadata.json"
    subtitles_file = "/home/znyd/hacking/edu-cut/src/pre_processing/segment_subtitle.json"
    output_file = "combined_output.json"

   
    # --- Run the main combination function ---
    combine_json_data(frames_file, subtitles_file, output_file)

    # --- Print the content of the final combined file ---
    print(f"\n--- Content of '{output_file}' ---")
    if os.path.exists(output_file):
        with open(output_file, 'r') as f:
            print(f.read())


Starting JSON combination process...
Successfully loaded frames data from '/home/znyd/hacking/edu-cut/src/pre_processing/vid_frames/segment_metadata.json'
Successfully loaded subtitles data from '/home/znyd/hacking/edu-cut/src/pre_processing/segment_subtitle.json'

Successfully combined data into 'combined_output.json'

--- Content of 'combined_output.json' ---
[
    {
        "time_stamps": "0s-10s",
        "frames": [
            "frame_00000.png",
            "frame_00001.png",
            "frame_00002.png",
            "frame_00003.png"
        ],
        "subtitle": [
            "The fundamental, root-of-it-all building block for linear algebra is the vector."
        ]
    },
    {
        "time_stamps": "10s-20s",
        "frames": [
            "frame_00004.png",
            "frame_00005.png",
            "frame_00006.png",
            "frame_00007.png",
            "frame_00008.png",
            "frame_00009.png",
            "frame_00010.png",
            "frame_00011.png",